In [13]:
from openai import OpenAI
from qdrant_client import QdrantClient
from dotenv import load_dotenv
from langsmith import traceable, get_current_run_tree

In [3]:
load_dotenv("../.env")

False

In [4]:
model = "text-embedding-3-small"
collection_name="Amazon-shopping-collection-01"
qdrant_client = QdrantClient(url="http://localhost:6333")
model_name = "gpt-5.4-nano"
openai = OpenAI()

In [15]:
@traceable( 
    name = 'get_embedding',
        run_type="embedding",
        metadata={
            "ls_model_provider": "openai",
            "ls_model_name": "text-embedding-3-small"
        }
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

In [6]:
@traceable
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(collection_name=collection_name, query=query_embedding, limit=k)
    context_ids = []
    context = []
    scores = []
    ratings = []

    print(results)
    for pt in results.points:
        context_ids.append(pt.payload['parent_asin'])
        context.append(pt.payload['preprocessed_description'])
        scores.append(pt.score)
        ratings.append(pt.payload['average_rating'])
        
    return {
        'retrieved_context_ids': context_ids,
        'retrieved_context': context,
        'similarity_scores': scores,
        'retrieved_context_ratings': ratings
    }

In [7]:
@traceable
def process_context(context):
    formated_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
        formated_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formated_context

In [8]:
@traceable
def build_prompt(context, query):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - answer the question based on the provided context only.
    - never use word context and refer to it as the available products.
    - do not use markdown formatting.

    Context:
    {context}

    Question:
    {query}
    """

    return prompt

In [19]:
@traceable(run_type="llm", metadata={'ls_model_provider':'openai', 'ls_model_name':'gpt-5.4-nano'})
def generate_answer(prompt):
    
    response = openai.chat.completions.create(
            model=model_name,
            messages=[{"role":"system","content": prompt}]
                     )
    
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"]={
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }

    return response.choices[0].message.content

In [10]:
@traceable
def rag_pipeline(question, top_k= 5):
    context = retrieve_data(question, top_k)
    preprocessed_context = process_context(context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    return answer

In [20]:
rag_pipeline("give me top few DVD Drive?", top_k=5)

points=[ScoredPoint(id=28, version=1, score=0.4768792, payload={'preprocessed_description': 'Original DVD Drive Replacement for Nintendo Wii Game Console Plug-and-Play Unit', 'image': 'https://m.media-amazon.com/images/I/51IbdwUfBLL._AC_.jpg', 'rating_number': 563, 'price': 29.41, 'average_rating': 4.7, 'parent_asin': 'B00APTQR62'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=10, version=1, score=0.43392196, payload={'preprocessed_description': "Gueray External CD DVD Drive for Laptop with USB 3.0 & Type-C Converter CD DVD Drive Portable CD DVD RW Reader Re-Writer Burner for Laptop Desktop PC Compatible with Windows Mac OS Vista Linux System (Rhombus shape)【High Speed via USB 3.0 & Type-C】Gueray external DVD drive supports the latest USB 3.0 technology to achieve the faster data transfer rate and more stable performance with the highest theoretical rate that is up to 5 GBPS. It is also compatible with USB 2.0 / 1.0, comes with Type-C converter, which is compatible wi

'Here are the top DVD drive options from the available products:\n\n1) Original DVD Drive Replacement for Nintendo Wii Game Console (ID: B00APTQR62) — Rating 4.7\n2) Gueray External CD/DVD Drive for Laptop/PC (USB 3.0 & Type-C) (ID: B094QQHJP7) — Rating 4.2\n\nIf you tell me whether you need an external drive for a laptop/PC or an internal replacement for a specific console, I can narrow it down further.'